# Quantum-Enhanced Agrivoltaics — Colab Simulation Launcher

**Manuscript:** Quantum-Coherent Spectral Engineering (jz-2026-00994t)  
**Solver:** PT-HOPS/SBD via MesoHOPS  

### Usage
- **Google Colab VS Code Extension**: open this notebook in VS Code with the Colab extension active. The runtime is remote; cells run on Colab's GPU/CPU.
- **Standalone Colab**: upload to [colab.research.google.com](https://colab.research.google.com) and run all cells.
- **Local (MesoHOP-sim env)**: `mamba run -n MesoHOP-sim jupyter notebook`

> ⚠️ **OOM note**: 100 trajectories at L=10 requires ~10 GB RAM per trajectory peak. On Colab (~12 GB), use `N_TRAJ = 5` for a quick test. For publication-grade results run on local hardware or HPC.

In [ ]:
# ── Cell 1: Environment detection & dependency install ──────────────────────
import sys, os, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('🌐 Google Colab detected — installing dependencies...')
    # Clone the repository (needed when not using the VS Code extension file sync)
    if not os.path.exists('Quantum_Agrivoltaic_PT-HOPS'):
        subprocess.check_call(['git', 'clone', '--depth=1',
            'https://github.com/NanaEngo/Quantum_Agrivoltaic_PT-HOPS.git'])
    os.chdir('Quantum_Agrivoltaic_PT-HOPS')

    # Install all required packages
    pkgs = [
        'numpy>=2.0,<2.4',   # numba constraint
        'scipy>=1.10',
        'pandas>=2.0',
        'matplotlib>=3.7',
        'pyyaml>=6.0',
        'numba>=0.59',       # required by MesoHOPS
        'h5py>=3.7',
        'tqdm>=4.65',
        'psutil>=5.9',
        'scienceplots>=2.0',
        'qutip>=5.2',
        'git+https://github.com/mesohops/mesohops.git@master',
    ]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
    print('✅ Dependencies installed.')
else:
    print('💻 Local environment detected — ensure MesoHOP-sim is active.')

# ── Path setup ───────────────────────────────────────────────────────────────
# Navigate to quantum_simulations_framework/ regardless of where notebook was opened
_root = os.getcwd()
_fw = os.path.join(_root, 'Redac_Paper1', 'quantum_simulations_framework')
if os.path.isdir(_fw):
    os.chdir(_fw)
elif 'quantum_simulations_framework' not in _root:
    raise RuntimeError(f'Cannot locate quantum_simulations_framework/ from {_root}')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'📂 Working directory: {os.getcwd()}')

In [ ]:
# ── Cell 2: Load & validate parameters.yaml ──────────────────────────────────
import yaml

with open('parameters.yaml') as f:
    config = yaml.safe_load(f)

print(f"Project : {config['metadata']['project']}")
print(f"L       : {config['dynamics']['hierarchy_depth']}")
print(f"K       : {config['dynamics']['matsubara_truncation']}")
print(f"Δt      : {config['dynamics']['time_step']} fs")
print(f"T       : {config['bath']['temperature']} K")
print(f"n_traj  : {config.get('simulation', {}).get('n_traj', 100)}")

In [ ]:
# ── Cell 3: Override n_traj for Colab (reduce to avoid OOM) ─────────────────
# Change N_TRAJ to 100 for publication-grade results (requires ~10 GB RAM/traj)
N_TRAJ = 5  # safe for Colab (~12 GB RAM); use 100 on local hardware

config.setdefault('simulation', {})['n_traj'] = N_TRAJ
print(f'n_traj set to {N_TRAJ} for this run.')

In [ ]:
# ── Cell 4: Check MesoHOPS availability ──────────────────────────────────────
from reproducibility.main import check_environment
assert check_environment(), '❌ MesoHOPS not available — re-run Cell 1'

In [ ]:
# ── Cell 5: Convergence audit (L=9, 10, 11) ──────────────────────────────────
# Uncomment to run; takes ~3 min on Colab
# from reproducibility.main import run_convergence_audit
# audit_data = run_convergence_audit()
# print(f"MAE(L10→L11): {audit_data['audit_mae_10_11']:.2e}")
print('ℹ️  Convergence audit skipped (uncomment to run).')

In [ ]:
# ── Cell 6: Full FMO simulation (filtered + broadband ensemble) ──────────────
from reproducibility.main import run_full_fmo_simulation
sim_results, time_points = run_full_fmo_simulation(config)
print('✅ Simulation complete.')

In [ ]:
# ── Cell 7: Generate & display figures ───────────────────────────────────────
from reproducibility.main import generate_figures
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

generate_figures(config, sim_results, time_points)

jpcl_dir = os.path.abspath('../../Theory_Journals/JPCL')
for pattern, title in [
    ('Quantum_dynamics_*.png', 'Figure 1 — Quantum Dynamics'),
    ('ETR_Under_Environmental_Effects*.png', 'Figure 2 — Environmental Robustness'),
]:
    files = sorted(glob.glob(os.path.join(jpcl_dir, pattern)))
    if files:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.imshow(mpimg.imread(files[-1]))
        ax.axis('off')
        ax.set_title(title)
        plt.tight_layout()
        plt.show()
    else:
        print(f'⚠️  {title} not found in {jpcl_dir}')